# 01 · Walkthrough: an E. coli run in the BioFlo 120

This notebook runs FermentIQ one step at a time on a batch record, so you can see (and explain in an interview) where every number comes from.

> **Privacy:** by default this uses the anonymized public sample. If you switch to your own record, clear all outputs before committing (VS Code: *Clear All Outputs*).

In [ ]:
import glob, os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project folder so relative paths work

from fermentiq.parse_record import parse_record
from fermentiq.checks import run_checks
from fermentiq.kinetics import analyze, find_mu_max
from fermentiq.html_report import write_html_report

PDF = "samples/sample_batch_record.pdf"  # anonymized public sample
# PDF = sorted(glob.glob("data/private/raw/*.pdf"))[0]  # or your own record (private)
PDF

## Step 1: Read the PDF
`parse_record` finds the three tables in the PDF (process, pellet/Gram stain, OD600). It turns messy entries like `1.5267g` or `T= 361 min` into numbers, and keeps the raw text for the documentation checks.

In [ ]:
record = parse_record(PDF)
print(record.header)
record.process[["sample", "raw_time_point", "clock_min", "temp_c", "ph_probe", "ph_meter", "used"]]

In [ ]:
record.od[["sample", "raw_time_point", "a_raw_1", "df_1", "a_corrected_1", "a_diluted_1"]]

## Step 2: Check the record like a QA reviewer
Every check in `checks.py` returns *findings*:
- 🔴 **critical**: a recorded number is wrong
- 🟠 **major**: a procedure or documentation gap
- 🟡 **minor**: a consistency or style issue
- 🔵 **info**: needs a human to look

In [ ]:
findings = run_checks(record)
findings[["severity", "check", "sample", "recorded", "expected"]]

### ✏️ Your turn
Verify one of the critical findings by hand: compute sample 3's pellet weight from the two tube weights. Does it match the 0.0174 g written on the record?

In [ ]:
p = record.pellets.set_index("sample")
p.loc[3, "tube_pellet_g"] - p.loc[3, "tube_g"]

## Step 3: Growth kinetics
During exponential growth, $OD = OD_0 e^{\mu t}$, so $\ln(OD) = \ln(OD_0) + \mu t$. **The slope of ln(OD) vs time is μ.**

`find_mu_max` slides a window over the points and keeps the steepest straight stretch (R² ≥ 0.95).

In [ ]:
kin = analyze(record)
print(kin["mu_max"])
kin["rates"]

### ✏️ Your turn
1. Try `min_points=2`. Why is μmax higher, and why is a 2-point estimate less trustworthy?
2. Try `min_r2=0.99` or `0.90`. How sensitive is the answer?

Write what you find in `LEARNING.md`.

In [ ]:
find_mu_max(kin["table"], min_points=2, min_r2=0.95)

## Step 4: Write the report
Creates the themed HTML report in `data/private/reports/` (never uploaded to GitHub). Try another theme: `"dark"`, `"print"`, `"lab"` or `"modern"`.

In [ ]:
path = write_html_report(record, findings, kin, theme="classic")
print(path)  # open this file in your browser